# Hiring and Firing

In the slides we saw that going into a strategy and hiring a manager are the same problem.

You need three things: a goal, how sure you want to be, and time.

Today lets try it on a manager everybody knows.

Suppose it is January 1999. You run manager selection at a pension fund, and you are thinking
about giving money to Warren Buffett.

- Would his record have convinced you?
- Once you hire him, when would you fire him?
- And what does it mean if the rule fires him at the wrong time?

**The plan**

1. Would you hire him? His record against the hiring line
2. You hired him. Now the clock restarts: his record against the walk-away line
3. What happened next. Then back to the slides
4. 🎯 Challenge (homework, after the rest of the slides): what if you trusted your process?

📑 **Slides for this class:** [Hiring and Firing (PDF)](https://amoreira2.github.io/UG54/slides/hiring_and_firing.pdf)

<iframe src="../../slides/hiring_and_firing.pdf" width="100%" height="520" style="border: 1px solid #ddd;"></iframe>

---

## Our data

Same daily data we used in the multi-factor lecture: Berkshire Hathaway and the Fama-French
factors. We compound the daily returns within each month: the same `(1+r).prod()-1` from the first lecture.

In [ ]:
#@title Setup — run this first
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy.stats import norm
%matplotlib inline
plt.rcParams['figure.figsize'] = [11, 4]

BASE = "https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data"

daily = pd.read_parquet(f"{BASE}/daily_funds_etfs.parquet")
daily = daily[['BRK', 'Mkt-RF', 'RF']].dropna()

# compound the days in each month
monthly = pd.DataFrame({
    'BRK': (1 + daily['BRK']).resample('ME').prod() - 1,
    'MKT': (1 + daily['Mkt-RF'] + daily['RF']).resample('ME').prod() - 1,
    'RF':  (1 + daily['RF']).resample('ME').prod() - 1,
}).loc['1989-01':]

brk_ex = monthly['BRK'] - monthly['RF']     # Berkshire's excess return
mkt_ex = monthly['MKT'] - monthly['RF']     # the market's excess return

print(f"{len(monthly)} months, {monthly.index[0]:%Y-%m} to {monthly.index[-1]:%Y-%m}")

The record we care about is Berkshire's **appraisal ratio** against the market: alpha over
the volatility of what the market does not explain.

Recall that after $T$ years an appraisal ratio has a standard error of about $1/\sqrt{T}$.

The function below re-estimates the CAPM regression every month, each time with one more month
of data, and saves the appraisal ratio. It waits for one year of data before it starts.

In [ ]:
def track_record(start, end=None):
    """Berkshire's appraisal ratio against the market, re-estimated every month from `start` on."""
    y_all = brk_ex.loc[start:end]
    x_all = mkt_ex.loc[start:end]
    rows = []
    for i in range(12, len(y_all) + 1):
        y = y_all.iloc[:i]
        X = sm.add_constant(x_all.iloc[:i])
        reg = sm.OLS(y, X).fit()
        alpha = reg.params['const'] * 12                 # per year
        resid_vol = reg.resid.std() * np.sqrt(12)        # per year
        rows.append({'date': y.index[-1], 'years': i / 12, 'AR': alpha / resid_vol})
    return pd.DataFrame(rows).set_index('date')

---

## 1. Would you hire him?

Recall the rule from the slides. You go in when the record clears

$$\widehat{AR}_T \;\geq\; a^* + \frac{z_p}{\sqrt{T}}$$

- $a^*$ is your goal: the appraisal ratio you are looking for
- $z_p$ is how sure you want to be: 1.64 for 95%
- $T$ is the number of years in the record

Lets start simple. You just want to be 95% sure that Buffett has a positive alpha, so $a^*=0$.

Before you start: how many years of Buffett do you think it takes?

The slider below is time. It starts after his first year, December 1989. The black line is
everything you would have seen up to that month; the dashed line is the bar. Move it forward one
year at a time and watch the two lines. (Click the slider, then the arrow keys move it one month.)

The other two sliders are the other two levers: your goal and how sure you want to be.

In [ ]:
from ipywidgets import interact, SelectionSlider, FloatSlider, Dropdown

record = track_record('1989-01', '1998-12')    # his record as of every month, up to the day you decide

def hire_view(date, goal, confidence):
    z = norm.ppf(confidence)
    hire_line = goal + z / np.sqrt(record['years'])
    seen = record.loc[:date]
    ar, years, bar = seen['AR'].iloc[-1], seen['years'].iloc[-1], hire_line.loc[date]

    fig, ax = plt.subplots()
    ax.plot(seen.index, seen['AR'], color='black', label='Berkshire record')
    ax.plot(hire_line.index, hire_line, color='tab:red', linestyle='--', label=f'hire line: goal {goal}, {confidence:.0%} sure')
    ax.scatter(date, ar, color='black', zorder=3)
    ax.set_xlim(pd.Timestamp('1989-01-01'), record.index[-1]); ax.set_ylim(0, 2.5)
    ax.set_ylabel('appraisal ratio'); ax.legend(loc='upper right'); plt.show()
    print(f"{date:%b %Y}: {years:.1f} years of record, appraisal ratio {ar:.2f}, hire line {bar:.2f}  ->  "
          + ("clears the line" if ar >= bar else "not yet"))

interact(hire_view,
         date=SelectionSlider(options=[(f"{d:%b %Y}", d) for d in record.index], description='record up to',
                              style={'description_width': 'initial'}, layout={'width': '600px'}),
         goal=FloatSlider(value=0, min=0, max=1, step=0.25, description='goal a*'),
         confidence=Dropdown(options=[0.80, 0.90, 0.95, 0.99], value=0.95, description='how sure'));

- After one year his appraisal ratio is 1.34. That is a fantastic number. Is it enough? Why not?
- Keep sliding. What happens to his record in 1990–1992? And to the line?
- When does the record first clear the line? How many years of data did you need?
- Is he above the line in December 1998, when you make the decision?
- Now go back to the first year and set `how sure` to 80%. Would you have hired him? What happened next?
- Set the goal to 0.5. Can you ever be 95% sure that Buffett is a 0.5 manager?
- In the slides, 5 in every 100 managers with no skill clear a 95% bar. Is that a reason to doubt
  Buffett's record? What else would you want to know?
- What does this tell you about how long it takes to learn about alpha?

---

## 2. You hired him. Now the clock restarts

Lets say you hired him in January 1999.

Recall from the slides: his record before that is why he is sitting in front of you. It is the estimate sample.
**The test starts the day you hire.** So from now on we only look at returns from January 1999 on.

You walk away once you are sure enough that he does not meet your goal:

$$\widehat{AR}_T \;\leq\; a^* - \frac{z_p}{\sqrt{T}}$$

Same three ingredients. The slider is time again: it starts after his first year with you,
December 1999. Slide forward and watch the record against the line.

In [ ]:
after = track_record('1999-01')     # his record since you hired him, as of every month
first = after.loc[:'2003-12']       # we look at his first five years with you

def fire_view(date, goal, confidence):
    z = norm.ppf(confidence)
    walk_away_line = goal - z / np.sqrt(first['years'])
    seen = first.loc[:date]
    ar, years, bar = seen['AR'].iloc[-1], seen['years'].iloc[-1], walk_away_line.loc[date]
    below = seen[seen['AR'] <= walk_away_line.loc[:date]]     # months where the record is below the line

    fig, ax = plt.subplots()
    ax.plot(seen.index, seen['AR'], color='black', label='Berkshire record since you hired him')
    ax.plot(walk_away_line.index, walk_away_line, color='tab:red', linestyle='--', label=f'walk-away line: goal {goal}, {confidence:.0%} sure')
    ax.scatter(date, ar, color='black', zorder=3)
    if len(below):
        ax.scatter(below.index[0], below['AR'].iloc[0], color='tab:red', marker='x', s=80, zorder=4, label='you walk away')
    ax.axhline(0, color='grey', linewidth=0.8)
    ax.set_xlim(pd.Timestamp('1999-01-01'), first.index[-1]); ax.set_ylim(-2.5, 1)
    ax.set_ylabel('appraisal ratio'); ax.legend(loc='lower right'); plt.show()
    print(f"{date:%b %Y}: {years:.1f} years with you, appraisal ratio {ar:.2f}, walk-away line {bar:.2f}  ->  "
          + (f"you walked away in {below.index[0]:%b %Y}" if len(below) else "keep him"))

interact(fire_view,
         date=SelectionSlider(options=[(f"{d:%b %Y}", d) for d in first.index], description='record up to',
                              style={'description_width': 'initial'}, layout={'width': '600px'}),
         goal=FloatSlider(value=0, min=0, max=1, step=0.25, description='goal a*'),
         confidence=Dropdown(options=[0.80, 0.90, 0.95, 0.99], value=0.95, description='how sure'));

- When does the rule fire him? How many months of record did it take?
- What was going on in 1999? Look at Berkshire and the market that year (next cell).
- Which lever moves the fire date the most: the goal or the confidence?
- Keep sliding after he is gone. What happens to his record?

In [ ]:
y1999 = (1 + monthly.loc['1999']).prod() - 1
print(f"1999:  Berkshire {y1999['BRK']:+.1%}   market {y1999['MKT']:+.1%}")

---

## 3. What happened next

Here is what you would have seen if you had kept him. The red line is the month the rule
(goal 0, 95% sure) fires him.

In [ ]:
goal, confidence = 0, 0.95
after['walk_away_line'] = goal - norm.ppf(confidence) / np.sqrt(after['years'])
fire_date = after[after['AR'] <= after['walk_away_line']].index.min()

window = monthly.loc['1999-01':'2003-12']
growth = (1 + window[['BRK', 'MKT']]).cumprod()
ax = growth.plot(color=['black', 'tab:blue'])
ax.axvline(fire_date, color='tab:red', linestyle='--', label=f'fired, {fire_date:%b %Y}')
ax.set_ylabel('growth of $1 invested in January 1999'); ax.legend(['Berkshire', 'market', f'fired, {fire_date:%b %Y}'])
plt.show()

after_fire = monthly.loc[fire_date + pd.offsets.MonthEnd(1):'2002-12']
print(f"from {after_fire.index[0]:%Y-%m} to 2002-12:  Berkshire {(1 + after_fire['BRK']).prod() - 1:+.1%}"
      f"   market {(1 + after_fire['MKT']).prod() - 1:+.1%}")

kept  = (1 + monthly.loc['1999-01':, 'BRK']).prod()
fired = (1 + monthly.loc['1999-01':fire_date, 'BRK']).prod() * (1 + monthly.loc[fire_date + pd.offsets.MonthEnd(1):, 'MKT']).prod()
print(f"$1 given to Buffett in January 1999, by {monthly.index[-1]:%Y-%m}:  kept ${kept:.2f}   fired and moved to the market ${fired:.2f}")

So was the rule wrong?

- Does one bad outcome tell you the rule was bad? Think about the Melvin example from the slides.
- The rule was 95% sure that Buffett was not a positive-alpha manager. Why was it so sure, after only
  one year?
- What did the rule ignore that you knew in January 1999?
- If you had been the one making this call, what would you have done?

---

## ⏸ Back to the slides

Stop here. Bring one answer back with you: **what did the rule not know?**

The challenge below is homework, for after class. It uses the second half of today's slides.

---

## 🎯 Challenge: what if you trusted your process?

*Homework — due before the next class.*

In the second half of the slides ("How much two flat years should move you") we had another way
to fire: be a Bayesian.

- There are two kinds of manager: good, with appraisal ratio $a$, and unskilled, with zero
- A share $\pi$ of the people your process hires are good. That is your hit rate: how much you trust your process
- Every year of record updates the odds that he is good:

$$\frac{\pi_T}{1-\pi_T} \;=\; \frac{\pi}{1-\pi} \times \exp\!\big(T\,a\,(\widehat{AR}_T - a/2)\big)$$

- Fire once the odds fall below 1: he is now more likely unskilled than good. Take logs and you get the fire line:

$$\widehat{AR}_T \;<\; \frac{a}{2} \;-\; \frac{1}{a\,T}\,\ln\frac{\pi}{1-\pi}$$

You hired Buffett to deliver $a = 0.5$. Use the record since January 1999, the same `after` as in
part 2. Note that $T$ here is the number of years since you hired him, the `years` column.

### Q1 — An investor with no inside view

Set $\pi = 0.5$. Does the rule fire him? After how many months of record?

> **📌 Required variable names:**
> ```python
> fires_50 = ____            # True or False
> months_to_fire_50 = ____   # months of record when it first fires, counting January 1999 as month 1
> ```

In [ ]:
# Your work here


fires_50 = ____
months_to_fire_50 = ____

print(fires_50, months_to_fire_50)

### Q2 — An investor who trusts the process

Now set $\pi = 0.8$: four out of five of the people you hire turn out to be good. Does the rule fire
him at any point up to December 2021?

> **📌 Required variable names:**
> ```python
> fires_80 = ____            # True or False
> ```

In [ ]:
# Your work here


fires_80 = ____

print(fires_80)

### Q3 — What each investor ended up with

\$1 given to Buffett in January 1999. One investor keeps him until December 2021. The other fires him
when the $\pi = 0.5$ rule fires, and moves the money to the market for the rest of the period.

> **📌 Required variable names:**
> ```python
> wealth_kept = ____         # $ value in December 2021, kept
> wealth_fired_50 = ____     # $ value in December 2021, fired by the pi = 0.5 rule
> ```

In [ ]:
# Your work here


wealth_kept = ____
wealth_fired_50 = ____

print(f"kept ${wealth_kept:.2f}   fired ${wealth_fired_50:.2f}")

### Q4 — The memo

> **📝 Your task — maximum eight sentences.**
>
> Your boss asks why two investors, looking at the same returns, made opposite calls on Buffett in
> early 2000. Explain what made the difference, and which investor you would rather have been.
> Then connect it to Citadel and Point72 putting money into Melvin in January 2021.
> Finally: would the investor who trusted the process have been just as right if Buffett had really
> lost his edge? What would have told them?

In [ ]:
MEMO = """
Write your memo here. Don't delete the surrounding triple quotes.
"""
print(MEMO)

---

## 📤 Submission

In [ ]:
# === 📤 SUBMISSION CELL — Run this last ===
import json, base64, hashlib, datetime as dt

required = ["fires_50", "months_to_fire_50", "fires_80", "wealth_kept", "wealth_fired_50", "MEMO"]
missing = [v for v in required if v not in globals()]
if missing:
    raise NameError(f"\n❌ Missing before submission: {missing}")

payload = {
    "assignment": "HiringAndFiring_AI",
    "ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip(),
}
blob = json.dumps(payload, sort_keys=True)
checksum = hashlib.sha256(blob.encode()).hexdigest()[:8]
token = f"UG54::{checksum}::{base64.b64encode(blob.encode()).decode()}"

print("=" * 72)
print("📋  COPY THE LINE BELOW AND PASTE INTO THE SUBMISSION FORM")
print("=" * 72)
print(token)
print("=" * 72)
print("Submission form: https://forms.gle/yazZ8bbatL87jdJi7")

---

## 🧠 Key Takeaways

- Hiring and firing are the same statistical decision: a goal, how sure you want to be, and time.
- Even Buffett needs years of record to clear a hiring line. You can never be 95% sure he is a 0.5 manager.
- Once you hire, the clock restarts. The record that got him hired is the estimate sample.
- A rule that only looks at returns can fire a great manager at the worst moment: Buffett in early 2000.
- How sure you need to be to fire depends on how much you trust the process that hired him.
- One outcome does not grade a decision. Melvin closed; Buffett did not. Both looked the same in their bad year.